# Ordered Logistic Regression Results for Adoption Predictors EDA with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All Croissant entities (record sets, fields, columns) are referenced by their `@id` values for consistency and reproducibility.

### Dataset Source
The dataset Croissant schema is available at:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's inspect all available record sets and their fields, using their `@id` values.

> **Note:** In Croissant, record sets represent logical tables or data groupings. Each record set has fields (`@id`) that correspond to dataset columns or keys. Let's enumerate them.

In [ ]:
# List record sets and their fields by @id
def print_record_sets(ds):
    if not hasattr(ds.metadata, "record_sets") or not ds.metadata.record_sets:
        print("No record sets defined in the metadata.")
        return
    for rs in ds.metadata.record_sets:
        print(f"RecordSet: {rs['@id']}")
        if 'fields' in rs:
            print("  Fields:")
            for f in rs['fields']:
                fname = f['name'] if 'name' in f else f['@id']
                print(f"    - {f['@id']} (name: {fname})")
        else:
            print("  No fields specified.")
        print("")

print_record_sets(dataset)

## 3. Data Extraction
Load data for the identified record sets into pandas DataFrames for further exploration and analysis.

To proceed, we need the actual `@id`s of the record sets. Let's fetch them automatically.

In [ ]:
# Extract all record set @ids
def get_record_set_ids(ds):
    if hasattr(ds.metadata, "record_sets") and ds.metadata.record_sets:
        return [rs["@id"] for rs in ds.metadata.record_sets]
    return []

record_sets_ids = get_record_set_ids(dataset)
print("Record set @ids found:", record_sets_ids)

# Load each record set to a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set '{record_set_id}'. Columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for record set '{record_set_id}'.")

# Display the head of the first available DataFrame
if dataframes:
    first_rs = next(iter(dataframes.keys()))
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We will now select a numeric field (column) and a grouping field for EDA:
- Filter records above a threshold
- Normalize the numeric field
- Group by a categorical field if present

> **Ensure to use field `@id` as the column name. Modify `numeric_field_id` and `group_field_id` as needed if structure is different for your dataset.

In [ ]:
# Select the first available DataFrame for demonstration
if not dataframes:
    print("No DataFrames were loaded. Please check the record set definitions in the Croissant schema.")
else:
    # We'll use the first record set and attempt analysis
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    print(f"Working with record_set_id: {record_set_id}")
    
    # Try to select a numeric field (heuristic: first field with numeric dtype or int/float in sample)
    numeric_field_id = None
    for col in df.columns:
        # Attempt conversion to numeric to check
        try:
            v = pd.to_numeric(df[col], errors='coerce')
            # If at least 50% values convert to number
            if v.notnull().sum() > 0.5 * len(df):
                numeric_field_id = col
                break
        except Exception:
            continue
    
    if numeric_field_id is None:
        print("Could not identify a numeric field in the first record set. Please specify manually.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Ensure numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # top 25% for demonstration
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {len(filtered_df)} records where '{numeric_field_id}' > {threshold:.2f}.")
        
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try a group field: pick first object dtype non-numeric col
        group_field = None
        for c in df.columns:
            if c != numeric_field_id and df[c].dtype == object:
                # Is probably a categorical
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped filtered data by '{group_field}':")
            print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric field and examine its relationship with the group field (if chosen in EDA).

Below, we use matplotlib and seaborn for basic visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data available for visualization.")
else:
    df = dataframes[record_set_id]
    # Check if we've already identified fields above
    if numeric_field_id is not None:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.show()

        if group_field:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=group_field, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.ylabel(numeric_field_id)
            plt.xlabel(group_field)
            plt.xticks(rotation=30, ha='right')
            plt.tight_layout()
            plt.show()
    else:
        print("No numeric field detected for plotting.")

## 6. Conclusion
This notebook demonstrated how to:
- Load and inspect a FAIR² Croissant dataset using `mlcroissant`
- Explore available record sets and fields using their `@id`s
- Extract and analyze records with pandas DataFrames
- Conduct basic EDA and visualize the results

**Key observations and further steps:**
- Check dataset field names and types when digging deeper. Use `@id` for unambiguous entity referencing.
- For more advanced analysis, refer to the full Croissant specification and the [mlcroissant documentation](https://github.com/mlcommons/croissant).

_End of notebook._